<a href="https://colab.research.google.com/github/keshav123333/langgraph/blob/main/lecture10_chatbot_basic_model.ipynb/lecture10_chatbot_basic_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain-huggingFace langgraph


In [3]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

llm=HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",


    temperature=0.7
)


# yaha maine kayde se like optimizer ke liye and generate tweet ke liye and evealuation teeno ke iye alag llm use karna chaiye jo uss task m excel ho
# like research ki kaunsa llm  vocab achi usse tweet generate kaunsa ache se feedback leta kaunsa sa llm evealuation achi karta aise

model=ChatHuggingFace(llm=llm)

# workflow

In [9]:
from langchain_core.messages import HumanMessage,SystemMessage,AIMessage,BaseMessage
from langgraph.graph import StateGraph, START, END
from typing import TypedDict,Literal,Annotated
import operator
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

In [10]:
class ChatState(TypedDict):
  message : Annotated[list[BaseMessage],add_messages] #yaha pe maine add_messages vo hi add wala kaam but like langgraph mein ye utility optimised for base message
  #base message ek class jisko human system ai mesaage sab inherit so iss list m vo sab ja sakte and addmessages unhe ache se add


In [11]:
def chat_node(state:ChatState):
  message=model.invoke(state.get("message"))
  return {"message":[message]}

In [13]:
checkpointer=MemorySaver()
graph = StateGraph(ChatState)

# add nodes
graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile(checkpointer=checkpointer)


In [14]:
thread_id='1'  #ham apne checkbot ke ek tread de rahe ek unique id de rahe hai
# abhi hum ram m save isko databse mein vaise save karte hai production m
while True:
  humanmsg=input("YOU : ")
  print("user " ,humanmsg)
  if humanmsg=="exit":
    break
  config={'configurable':{"thread_id":thread_id}}
  res=chatbot.invoke({"message":[HumanMessage(content=humanmsg)]},config=config)  # dekh isme hum [humanmessage] as jab ye graph ke andar jaeyga toh vo add hota jayega

  print("AI",res.get("message")[-1].content) ## as message ke andar last wala content hi reply hoga as chatnode me vahi store kiya hoga


YOU : hi my name is keshav
user  hi my name is keshav
AI Nice to meet you, Keshav. Is there something I can help you with or would you like to chat?
YOU : whats my name 
user  whats my name 
AI Your name is Keshav.
YOU : cool yaar kal college jaane ka man ni kar raha 
user  cool yaar kal college jaane ka man ni kar raha 
AI Aajkal college jaana kaisa hai? (College going these days is what?) Kya koi issue hai ya koi problem? Main sun sakta hoon.
YOU : exir
user  exir
AI Exir? Kya hoga? Kya problem hai? Bata, main samjh sakta hoon.
YOU : exit
user  exit


In [15]:
chatbot.get_state(config=config) # ye sari state de dega like abhi tak ke msg

StateSnapshot(values={'message': [HumanMessage(content='hi my name is keshav', additional_kwargs={}, response_metadata={}, id='ce21037f-9f66-4aa0-8cf0-07c0cec4e606'), AIMessage(content='Nice to meet you, Keshav. Is there something I can help you with or would you like to chat?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 42, 'total_tokens': 67}, 'model_name': 'meta-llama/Llama-3.1-8B-Instruct', 'system_fingerprint': 'fp_ff5ddd64106da7b20c62', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d3fb5-5811-7bc1-b52b-c56f912f78e4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 25, 'total_tokens': 67}), HumanMessage(content='whats my name ', additional_kwargs={}, response_metadata={}, id='6e95921d-b6e8-4d41-bc80-6781abb5a9d2'), AIMessage(content='Your name is Keshav.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 80, 'total_token